# Archetype PCA Analysis

This notebook builds a PCA map from management-aligned archetype features. It is intentionally separate from `archetype_classification.ipynb` so classification/scoring and PCA interpretation can be reviewed independently.

The PCA here should be treated as the **visual map**, while the archetype scores remain the **explanation layer**.

## Approach

1. Load D-I, D-II, and D-III source CSVs.
2. Build common archetype features in memory.
3. Convert those features to division-relative percentiles.
4. Fit PCA on the percentile features.
5. Inspect explained variance and component loadings.
6. Plot the PCA map colored by primary archetype.

No source datasets are modified.

## PCA Meaning Guide

The notebook includes a section named **What Each PCA Means** after **Component Loadings**. That section generates a `pca_meanings` table after PCA runs.

How to read it:
- A player's `PC1`, `PC2`, etc. score is their position on that axis.
- The axis meaning comes from the loadings table, not from the PC number itself.
- Strong positive loadings describe the high end of the axis.
- Strong negative loadings describe the low end of the axis.

Example: if PC1 has positive loadings for size and defensive rebounding, and negative loadings for assist creation and 3P volume, then higher PC1 means more big/rebound profile while lower PC1 means more guard/creator profile.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

HERE = Path.cwd()
D1_PATH = HERE / "mbb_with_pca.csv"
D2_PATH = HERE / "d2_data_cleaned.csv"
D3_PATH = HERE / "d3_data_cleaned.csv"

D1_PATH.exists(), D2_PATH.exists(), D3_PATH.exists()

## Load And Normalize Player Data

D-I has true `AST_pct` and `DRB_pct`. D-II/D-III use `ast_per_40` and `DRBPG` proxies because true assist/rebound rates are not available in the cleaned files.

In [ ]:
D1_ROLE_MAP = {
    "Pure PG": "G",
    "Scoring PG": "G",
    "Combo G": "G",
    "Wing G": "G/F",
    "Wing F": "F",
    "Stretch 4": "F/C",
    "PF/C": "F/C",
    "C": "C",
}

def numeric(series, default=0):
    return pd.to_numeric(series, errors="coerce").fillna(default)

def flip_name(value):
    value = "" if pd.isna(value) else str(value).strip()
    if "," not in value:
        return value
    last, first = value.split(",", 1)
    return f"{first.strip()} {last.strip()}".strip()

def normalize_position(value):
    value = "" if pd.isna(value) else str(value).strip().upper()
    if value in {"G", "G/F", "F", "F/C", "C"}:
        return value
    if value.startswith("GUARD") or value == "PG":
        return "G"
    if value.startswith("FORWARD"):
        return "F"
    if value.startswith("CENTER"):
        return "C"
    if value.startswith("G/F") or value.startswith("GF"):
        return "G/F"
    if value.startswith("F/C") or value.startswith("FC"):
        return "F/C"
    if value.startswith("G"):
        return "G"
    if value.startswith("F"):
        return "F"
    if value.startswith("C"):
        return "C"
    return "G"

def load_d1(path):
    raw = pd.read_csv(path)
    return pd.DataFrame({
        "division": "D-I",
        "source_file": path.name,
        "name": raw["player_name"].astype(str).str.strip(),
        "team": raw["team"].astype(str).str.strip(),
        "conference": raw["conf"].astype(str).str.strip(),
        "pos": raw["role"].map(D1_ROLE_MAP).fillna("G"),
        "role": raw["role"].astype(str),
        "height_in": numeric(raw["height_inches"], 78),
        "mpg": numeric(raw["mins_per_game"]),
        "ppg": numeric(raw["pts_per_game"]),
        "apg": numeric(raw["ast_per_game"]),
        "rpg": numeric(raw["treb_per_game"]),
        "three_pct": numeric(raw["3P_pct"]),
        "three_rate": numeric(raw["three_share"]),
        "efg": numeric(raw["eFG"]) / 100.0,
        "ast_tov": numeric(raw["AST_TOV"]),
        "assist_creation": numeric(raw["AST_pct"]),
        "assist_source": "AST_pct",
        "dreb": numeric(raw["DRB_pct"]),
        "dreb_source": "DRB_pct",
    })

def load_cleaned_division(path, division):
    raw = pd.read_csv(path)
    return pd.DataFrame({
        "division": division,
        "source_file": path.name,
        "name": raw["Player Name"].apply(flip_name),
        "team": raw["Team"].astype(str).str.strip(),
        "conference": raw["Conference"].astype(str).str.strip(),
        "pos": raw["Position"].apply(normalize_position),
        "role": raw["Position"].astype(str),
        "height_in": numeric(raw["Height"], 72),
        "mpg": numeric(raw["MPG"]),
        "ppg": numeric(raw["PPG"]),
        "apg": numeric(raw["APG"]),
        "rpg": numeric(raw["RPG"]),
        "three_pct": numeric(raw["3PT%"]),
        "three_rate": numeric(raw["three_share"]),
        "efg": numeric(raw["eFG"]),
        "ast_tov": numeric(raw["AST_TOV"]),
        "assist_creation": numeric(raw["ast_per_40"]),
        "assist_source": "ast_per_40 proxy",
        "dreb": numeric(raw["DRBPG"]),
        "dreb_source": "DRBPG proxy",
    })

players = pd.concat([
    load_d1(D1_PATH),
    load_cleaned_division(D2_PATH, "D-II"),
    load_cleaned_division(D3_PATH, "D-III"),
], ignore_index=True)

players["player_key"] = players["division"] + " | " + players["team"] + " | " + players["name"]
players.shape

## Build Archetype Percentile Features

In [ ]:
def percentile_by(group_cols, value_col):
    return players.groupby(group_cols)[value_col].rank(pct=True, method="average") * 100

players["pct_assist_creation"] = percentile_by("division", "assist_creation")
players["pct_three_pct"] = percentile_by("division", "three_pct")
players["pct_three_rate"] = percentile_by("division", "three_rate")
players["pct_ast_tov"] = percentile_by("division", "ast_tov")
players["pct_efg"] = percentile_by("division", "efg")
players["pct_dreb_pos_adj"] = percentile_by(["division", "pos"], "dreb")
players["pct_size"] = percentile_by("division", "height_in")

pca_features = [
    "pct_assist_creation",
    "pct_three_pct",
    "pct_three_rate",
    "pct_ast_tov",
    "pct_efg",
    "pct_dreb_pos_adj",
    "pct_size",
]

players[pca_features].describe().round(2)

## Rebuild Archetype Scores For Coloring

These scores are the same idea as the classification notebook: they give us a primary archetype for coloring the PCA map.

In [ ]:
players["meets_pg_preferred"] = (
    (players["pct_assist_creation"] >= 70)
    & (players["three_pct"] >= 0.330)
    & (players["three_rate"] >= 0.300)
    & (players["ast_tov"] > 1.000)
)

players["meets_wing_preferred"] = (
    (players["pct_dreb_pos_adj"] >= 60)
    & (players["three_pct"] >= 0.330)
    & (players["three_rate"] >= 0.300)
    & (players["ast_tov"] > 1.000)
)

players["meets_big_preferred"] = (
    (players["height_in"] >= 79)
    & (players["pct_dreb_pos_adj"] >= 60)
    & (players["three_pct"] >= 0.300)
    & (players["three_rate"] >= 0.250)
    & (players["ast_tov"] > 1.000)
)

players["score_pg_combo"] = (
    0.35 * players["pct_assist_creation"]
    + 0.20 * players["pct_three_pct"]
    + 0.15 * players["pct_three_rate"]
    + 0.20 * players["pct_ast_tov"]
    + 0.10 * players["pct_efg"]
    + np.where(players["meets_pg_preferred"], 8, 0)
).clip(0, 100)

players["score_wing_2_4"] = (
    0.30 * players["pct_dreb_pos_adj"]
    + 0.25 * players["pct_three_pct"]
    + 0.20 * players["pct_three_rate"]
    + 0.15 * players["pct_ast_tov"]
    + 0.10 * players["pct_size"]
    + np.where(players["meets_wing_preferred"], 8, 0)
).clip(0, 100)

players["score_stretch_big"] = (
    0.30 * players["pct_dreb_pos_adj"]
    + 0.25 * players["pct_size"]
    + 0.20 * players["pct_three_pct"]
    + 0.15 * players["pct_three_rate"]
    + 0.10 * players["pct_ast_tov"]
    + np.where(players["meets_big_preferred"], 8, 0)
).clip(0, 100)

score_cols = ["score_pg_combo", "score_wing_2_4", "score_stretch_big"]
archetype_labels = {
    "score_pg_combo": "PG / Combo Guard",
    "score_wing_2_4": "2-4 Interchangeable Wing",
    "score_stretch_big": "5 / Stretch 4 / Big Wing",
}

players["primary_score_col"] = players[score_cols].idxmax(axis=1)
players["primary_archetype"] = players["primary_score_col"].map(archetype_labels)
players["primary_score"] = players[score_cols].max(axis=1)

players["primary_archetype"].value_counts()

## Fit PCA

In [ ]:
X_raw = players[pca_features].fillna(players[pca_features].median()).to_numpy(dtype=float)
feature_means = X_raw.mean(axis=0)
feature_stds = np.where(X_raw.std(axis=0, ddof=0) == 0, 1, X_raw.std(axis=0, ddof=0))
X = (X_raw - feature_means) / feature_stds

try:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=4, random_state=0)
    coords = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_
    components = pca.components_
except Exception:
    # SVD fallback if sklearn is unavailable.
    u, s, vt = np.linalg.svd(X, full_matrices=False)
    coords = u[:, :4] * s[:4]
    eigenvalues = (s ** 2) / (len(X) - 1)
    explained = eigenvalues[:4] / eigenvalues.sum()
    components = vt[:4]

for i in range(4):
    players[f"arch_pca_PC{i + 1}"] = coords[:, i]

explained_df = pd.DataFrame({
    "component": [f"PC{i + 1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "cumulative_explained_variance": np.cumsum(explained),
})

explained_df.round(4)

## Component Loadings

Use this table to decide what each PCA axis actually means. Do not name `PC1` or `PC2` until these loadings are reviewed.

In [ ]:
loadings = pd.DataFrame(
    components.T,
    index=pca_features,
    columns=[f"PC{i + 1}" for i in range(components.shape[0])],
)

loadings.round(3)

## What Each PCA Means

PCA components do not come with built-in basketball names. The meaning comes from the feature loadings above: large positive loadings describe one side of the axis, and large negative loadings describe the other side.

The cell below creates an interpretable summary for each component using the strongest positive and negative loadings.

In [ ]:
feature_labels = {
    "pct_assist_creation": "assist creation",
    "pct_three_pct": "3P shooting quality",
    "pct_three_rate": "3P shooting volume",
    "pct_ast_tov": "ball security / A-to-TO",
    "pct_efg": "overall shooting efficiency",
    "pct_dreb_pos_adj": "position-adjusted defensive rebounding",
    "pct_size": "size / height",
}

def describe_component(component_name, n_terms=3):
    vals = loadings[component_name].sort_values(ascending=False)
    positive = vals.head(n_terms)
    negative = vals.tail(n_terms).sort_values()
    return pd.Series({
        "positive_side": ", ".join(f"{feature_labels[idx]} ({val:+.2f})" for idx, val in positive.items()),
        "negative_side": ", ".join(f"{feature_labels[idx]} ({val:+.2f})" for idx, val in negative.items()),
        "reading": (
            f"Higher {component_name} = more "
            + ", ".join(feature_labels[idx] for idx in positive.index)
            + "; lower = more "
            + ", ".join(feature_labels[idx] for idx in negative.index)
            + "."
        ),
    })

pca_meanings = pd.DataFrame({
    component: describe_component(component)
    for component in loadings.columns
}).T

pca_meanings

## All PCA Loading Values

This section shows every feature loading for every PC. The `abs_loading` column is the strength of that feature on the component, ignoring direction. Use this table as the factual base before writing any basketball interpretation.

In [ ]:
loading_values = (
    loadings.reset_index(names="feature")
    .melt(id_vars="feature", var_name="component", value_name="loading")
)

loading_values["feature_label"] = loading_values["feature"].map(feature_labels)
loading_values["abs_loading"] = loading_values["loading"].abs()

loading_values = loading_values.sort_values(
    ["component", "abs_loading"],
    ascending=[True, False],
).reset_index(drop=True)

loading_values.round(3)

## Data-Driven PCA Interpretation Draft

The cell below does not assign a final label like `shooting axis` or `guard axis`. It lists the largest loading values and gives a neutral reading based on those values. After reviewing the loading table above, you can rename the axis manually if the basketball meaning is clear.

In [ ]:
def prose_for_component(component_name, n_terms=7):
    vals = loadings[component_name].sort_values(key=lambda s: s.abs(), ascending=False)
    ordered = vals.head(n_terms)
    positive = ordered[ordered > 0]
    negative = ordered[ordered < 0]
    variance = explained_df.loc[explained_df["component"] == component_name, "explained_variance_ratio"].iloc[0] * 100

    positive_text = ", ".join(f"{feature_labels[idx]} ({val:+.2f})" for idx, val in positive.items()) or "no major positive loadings"
    negative_text = ", ".join(f"{feature_labels[idx]} ({val:+.2f})" for idx, val in negative.items()) or "no major negative loadings"
    all_values = "; ".join(f"{feature_labels[idx]} ({val:+.2f})" for idx, val in ordered.items())

    return (
        f"### {component_name}\n"
        f"{component_name} explains **{variance:.1f}%** of the variation in the archetype feature space. "
        f"The strongest positive values are: **{positive_text}**. "
        f"The strongest negative values are: **{negative_text}**.\n\n"
        f"All loading values used for this read: {all_values}.\n\n"
        f"Working interpretation: higher {component_name} scores lean toward the positive-loading traits above, while lower {component_name} scores lean toward the negative-loading traits. Review the full loading table before giving this axis a basketball label."
    )

pca_interpretation_text = "\n\n".join(
    prose_for_component(component)
    for component in loadings.columns
)

print(pca_interpretation_text)

The draft text above is intentionally cautious. The values are the important part; the final axis names should come after you inspect which loadings are meaningfully large and whether the direction makes basketball sense.

## Basketball Interpretation Of The PCA Axes

Based on the loading values above, these are the most useful basketball reads of the components.

### PC1: Size / Efficiency vs Guard Creation

PC1 explains about **33.7%** of the archetype variation. The positive side is driven mostly by **size / height**. The negative side is much stronger for **A-to-TO**, **assist creation**, **3P volume**, and **3P quality**.

Basketball read: higher PC1 players tend to look more like bigger, less guard-creation-oriented players. Lower PC1 players tend to look more like creator guards or perimeter initiators: better passing profile, better ball security, and more 3P involvement. Because the defensive rebounding loading is very small here, this axis is more about **size versus creator-guard skill** than true rebounding.

Suggested label: **Size vs Creator Guard Traits**.

### PC2: Shooting / Spacing Strength

PC2 explains about **21.3%** of the archetype variation. The positive side is clearly driven by **3P shooting quality**, **overall shooting efficiency**, and **3P volume**. The negative side has smaller loadings, mostly **assist creation** and a little **A-to-TO**.

Basketball read: higher PC2 players are the cleanest shooting and spacing fits. They combine make quality with volume and efficiency. Lower PC2 players are less defined by shooting, and may lean more toward creation or non-shooting utility.

Suggested label: **Shooting / Spacing Strength**.

### PC3: Defensive Rebounding + Connective Creation vs 3P Volume

PC3 explains about **19.6%** of the archetype variation. The positive side is strongest for **position-adjusted defensive rebounding**, with meaningful positives for **assist creation**, **efficiency**, and **A-to-TO**. The negative side is mainly **3P shooting volume**.

Basketball read: higher PC3 players look like rebound-capable connectors: players who rebound for their position while also showing some passing, efficiency, and ball-security value. Lower PC3 players are more defined by perimeter volume, especially 3P rate. This is not purely a guard axis or a size axis; it is more of a **connector/rebounder vs volume-spacer** split.

Suggested label: **Rebounding Connector vs 3P Volume**.

### PC4: Rebounding + 3P Volume vs Efficiency / Ball Security

PC4 explains about **10.2%** of the archetype variation, so it should be treated as a secondary/deeper-analysis axis. The positive side is mostly **position-adjusted defensive rebounding**, with some **3P volume** and **3P quality**. The negative side is **overall efficiency**, **A-to-TO**, **assist creation**, and slightly **size**.

Basketball read: higher PC4 players are rebound-and-volume-shooting profiles, but not necessarily the most efficient or secure decision-makers. Lower PC4 players are more efficiency/ball-security oriented. Since this component explains much less variance than PC1 and PC2, it is probably less important for the main dashboard view.

Suggested label: **Rebounding + 3P Volume vs Efficiency / Security**.

### Practical Dashboard Recommendation

Use **PC1 and PC2** as the primary map axes. Together they explain about **55%** of the archetype feature variation and are the easiest to explain:

- **PC1:** Size vs creator guard traits
- **PC2:** Shooting / spacing strength

Keep **PC3** and **PC4** in the notebook for deeper review. They are useful, but they are less intuitive and should probably not be the first thing coaches or management see.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
explained_df.set_index("component")["explained_variance_ratio"].plot(kind="bar", ax=ax, color="#4a9eed")
ax.set_title("Archetype PCA Explained Variance")
ax.set_ylabel("Explained variance ratio")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()

## PCA Map Colored By Archetype

In [ ]:
colors = {
    "PG / Combo Guard": "#4a9eed",
    "2-4 Interchangeable Wing": "#7cc47a",
    "5 / Stretch 4 / Big Wing": "#e8a44a",
}

fig, ax = plt.subplots(figsize=(10, 7))

for archetype, sub in players.groupby("primary_archetype"):
    ax.scatter(
        sub["arch_pca_PC1"],
        sub["arch_pca_PC2"],
        s=22,
        alpha=0.65,
        label=archetype,
        color=colors.get(archetype, "#999999"),
        edgecolors="none",
    )

ax.axhline(0, color="#cccccc", linewidth=0.8)
ax.axvline(0, color="#cccccc", linewidth=0.8)
ax.set_title("Archetype PCA Map")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.legend(frameon=False)
plt.tight_layout()

## PCA Map By Division

This helps check whether the PCA is separating archetypes or simply separating data sources/divisions.

In [ ]:
division_markers = {"D-I": "o", "D-II": "s", "D-III": "^"}

fig, ax = plt.subplots(figsize=(10, 7))

for division, sub in players.groupby("division"):
    ax.scatter(
        sub["arch_pca_PC1"],
        sub["arch_pca_PC2"],
        s=22,
        alpha=0.55,
        label=division,
        marker=division_markers.get(division, "o"),
    )

ax.axhline(0, color="#cccccc", linewidth=0.8)
ax.axvline(0, color="#cccccc", linewidth=0.8)
ax.set_title("Archetype PCA Map By Division")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.legend(frameon=False)
plt.tight_layout()

## Top PG / Combo Fits On The PCA Map

In [ ]:
top_pg = players.sort_values("score_pg_combo", ascending=False).head(25).copy()

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(players["arch_pca_PC1"], players["arch_pca_PC2"], s=12, alpha=0.15, color="#888888")
ax.scatter(top_pg["arch_pca_PC1"], top_pg["arch_pca_PC2"], s=55, alpha=0.9, color="#4a9eed")

for _, row in top_pg.head(12).iterrows():
    ax.annotate(row["name"], (row["arch_pca_PC1"], row["arch_pca_PC2"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.axhline(0, color="#cccccc", linewidth=0.8)
ax.axvline(0, color="#cccccc", linewidth=0.8)
ax.set_title("Top PG / Combo Fits Highlighted")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
plt.tight_layout()

top_pg[[
    "division", "name", "team", "pos", "height_in", "three_pct", "three_rate", "ast_tov",
    "assist_creation", "assist_source", "score_pg_combo", "primary_archetype", "arch_pca_PC1", "arch_pca_PC2",
]].round(3)

## Optional Export

Disabled by default. Turning this on writes a separate PCA result file and does not modify source datasets.

In [ ]:
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    export_cols = [
        "division", "name", "team", "conference", "pos", "height_in", "mpg", "ppg", "apg", "rpg",
        "three_pct", "three_rate", "efg", "ast_tov", "assist_creation", "assist_source", "dreb", "dreb_source",
        "primary_archetype", "primary_score", "score_pg_combo", "score_wing_2_4", "score_stretch_big",
        "arch_pca_PC1", "arch_pca_PC2", "arch_pca_PC3", "arch_pca_PC4",
    ]
    players.loc[:, export_cols].to_csv(HERE / "archetype_pca_results.csv", index=False)
    loadings.to_csv(HERE / "archetype_pca_loadings.csv")
    print("Wrote archetype_pca_results.csv and archetype_pca_loadings.csv")
else:
    print("EXPORT_RESULTS is False; no files written.")